In [1]:
import pandas as pd
from datetime import datetime, timedelta
from my_preprocess_fn import add_holidays_nyc, add_holidays_tky, make_uppercat
from file_reader import FileReader

# 뉴욕 데이터 전처리

In [2]:
nyc_df = FileReader.read_dataset(file_name='dataset_TSMC2014_NYC.txt', dataset_name='nyc')

nyc_df = FileReader.do_filter(nyc_df, poi_min_freq=10, user_min_freq=10)

nyc_df = FileReader.split_train_test(nyc_df)

nyc_df = add_holidays_nyc(nyc_df)

nyc_df = make_uppercat(nyc_df, opt='NYC')

# 경로 생성
nyc_df = FileReader.generate_traj_id(nyc_df)

In [6]:
print("NYC DataFrame shape:", nyc_df.shape)
print(f"num_users: {nyc_df['UserId'].nunique()}, num_pois: {nyc_df['PoiId'].nunique()}, num_trajectories: {nyc_df['TrajectoryId'].nunique()}")

NYC DataFrame shape: (116425, 19)
num_users: 1081, num_pois: 4637, num_trajectories: 26924


In [15]:
def build_prev_traj_mapping(df: pd.DataFrame):
    """
    TrajectoryId = {user_id}_{traj_n} 형태일 때,
    traj_n >= 2 인 trajectory에 대해 바로 직전 trajectory id를 매핑해줌.

    return: dict {traj_id: prev_traj_id}
    """
    mapping = {}

    # TrajectoryId 분리
    for traj_id in df['TrajectoryId'].unique():
        try:
            user_id, traj_n = traj_id.split("_")
            traj_n = int(traj_n)
        except ValueError:
            continue  # 형식이 맞지 않는 경우 skip

        if traj_n >= 2:
            prev_traj_id = f"{user_id}_{traj_n-1}"
            mapping[traj_id] = prev_traj_id

    return mapping

In [16]:
# prev_traj_mapping = build_prev_traj_mapping(nyc_df)
# prev_traj_mapping


In [17]:
# 데이터 분할
nyc_train = nyc_df[nyc_df['SplitTag'] == 'train'] 
nyc_val = nyc_df[nyc_df['SplitTag'] == 'validation']
nyc_test = nyc_df[nyc_df['SplitTag'] == 'test']

In [18]:
# 저장
nyc_train.to_csv('../data/nyc/raw/NYC_train.csv', index=False)
nyc_val.to_csv('../data/nyc/raw/NYC_val.csv', index=False)
nyc_test.to_csv('../data/nyc/raw/NYC_test.csv', index=False)

In [19]:
# 전체 데이터 저장
nyc_df.to_csv('../data/nyc/raw/NYC_df.csv', index=False)

# 도쿄 데이터 전처리

In [20]:
tky_df = FileReader.read_dataset(file_name='dataset_TSMC2014_TKY.txt', dataset_name='tky')

tky_df = FileReader.do_filter(tky_df, poi_min_freq=10, user_min_freq=10)

tky_df = FileReader.split_train_test(tky_df)

tky_df = add_holidays_tky(tky_df)

tky_df = make_uppercat(tky_df, opt='TKY')

tky_df = FileReader.generate_traj_id(tky_df)

In [21]:
# 데이터 분할
tky_train = tky_df[tky_df['SplitTag'] == 'train'] 
tky_val = tky_df[tky_df['SplitTag'] == 'validation']
tky_test = tky_df[tky_df['SplitTag'] == 'test']

# 저장
tky_train.to_csv('../data/tky/raw/TKY_train.csv', index=False)
tky_val.to_csv('../data/tky/raw/TKY_val.csv', index=False)
tky_test.to_csv('../data/tky/raw/TKY_test.csv', index=False)

In [22]:
# 전체 저장
tky_df.to_csv('../data/tky/raw/TKY_df.csv', index=False)